# Data Mining 2026  — Project Notebook

**Course:** Data Mining  
**Project track:** Standard Analysis  
**Group members:**  
- Mads Pagh
- Marcus Mofjeld

**GITHUB REPO**
- https://github.com/Zyzzava/datamining

**Dataset:**  Spotify Playlists

**Initial task description (Module 1 perspective):** We propose a project that critiques and expands upon the methodology presented in "Towards a Context-Aware Music Recommendation Approach: What is Hidden in the Playlist Name?" (Pichl et al., 2015). The original authors demonstrated that clustering playlist names to infer "listening contexts" can increase recommendation precision by 33% compared to traditional collaborative filtering. However, their approach relied exclusively on K-Means clustering. Our analysis goal in this first part is Structure Discovery, specifically investigating the intrinsic geometry of these "listening contexts." As noted in the course material, K-Means is a Representative-based algorithm that assumes clusters are spherical, compact, and distinct. We hypothesize that human-generated musical contexts are likely complex, arbitrary in shape,
and potentially nested, which makes the rigid partitioning of K-Means a limitation. To test this, we will compare
the K-Means baseline against alternative clustering paradigms. By comparing these techniques, we aim to determine which structural definition best captures the semantic reality of user playlists.

## Reproducibility and Setup

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Dataset Description and Loading

### Dataset Overview

- Source: https://www.kaggle.com/datasets/andrewmvd/spotify-playlists
- Number of instances: 12,902,045
- Number of features / entities: 4
- Missing values: 0
- Basic statistics:
Unique Users:            15,918
Unique Tracks:           2,036,899
Unique Raw Playlists:    157,530


Heal step is needed to fix some issues with the dataset, mainly human insertion errors, double quoutes, etc., disallowing CSV loading. 
- Only needed in case of reproducability check, otherwise just loading from `spotify_fully_processed.parquet`.

In [ ]:
from preprocessing.preprocessor import load_and_heal_data

# Load dataset (dataframe)
df = load_and_heal_data() 

In [ ]:
# View the first few rows of the dataframe (verification)
df.iloc[[10000, 300000, 470000], :4]

### Preprocessing
Our preprocessing pipeline aims to match and improve upon that of [CITATION?]. The pipeline consists of the following main steps on the playlist names column:
#### Homogenization 
Homogenization is the process of extracting the base form (lemma) of a word. The process consists of two main steps. Firstly we use regular expressions to make all letters lowercase and strip out punctuation. Next we break all the words in the names into tokens and lemmatize them (e.g., converting "workouts" to "workout"). Practically this is done using the Natural Language Toolkit (NLTK) library. The aim with this process is to transform the messy inconsistent user-generated names into a standard format, which makes string matching more accurate in the clustering algorithms.

In [ ]:
from preprocessing.preprocessor import homogenize_playlists
df = homogenize_playlists(df)
# displaying the effect of homogenization on playlistnames (verification)
df.iloc[[10000, 300000, 470000], 4]

#### Entity Filtering
The goal of entity filtering is to remove playlists that lack contextual information, or are strictly based on named entities (like specific artists, or bands). As we are trying to cluster based on the context described in the name, we filter out these non-contextual playlists to ensure our clustering models learn situational patterns rather than grouping artist discographies together.

To do this we run a strict filtering process, where we compare the playlist names against the artists present in the dataset, along with a set of predefined genres. To take it one step further, we use the spaCy NLP library to analyze the grammatical structure for further named entity recognition. The model looks for entities tagged as "person", "org" or "work of art", and removes playlists where this consists of 80% or more of the name. We also implement a "rescue mechanism", in which the script checks tokens for adjectives or verbs and ensures these playlists are kept, as the presence of an action or description implies context, regardless of the rest of the name.

In [ ]:
from preprocessing.preprocessor import filter_entities
df = filter_entities(df)
df.iloc[[10000, 300000, 470000], 5]

#### Stop-word Filtering
Stop-word filtering removes the words, which we do not want to expand and cluster on. Stop words define any words that are often used without providing semantic meaning. In our case we filter on both classic stop words such as "as" and "a" along with music specific stop words such as "track" and "playlist", which do not provide meaning to the playlist name. 

In [ ]:
from preprocessing.preprocessor import remove_stop_words
df = remove_stop_words(df)
df.iloc[[300000], 6]


#### Feature Expansion
The `expand_feature` function is responsible for the text preprocessing, which enriches the short strings, e.g. a playlist name, but appending related words to it. This technique provides more context for text-mining algorithms like TF-IDF. The flow of the function is the following:
1. First the input gets validated, an empty string is returned if not. 
2. A set is maintaned which stores the words, preventing duplicate entries. 
3. We then split the input strings into individual words and process them one by one. For each word, WordNet is used to find
    - Synonyms (Lemmas), which contain the exact same meaning. 
    - Hypernyms expanding general category words ('guitar' -> 'stringed instrument'). 
4. Lastly, the underscores from WordNet terms are replaced with spaces, and the final deduplicated set of original words, synonyms, and hypernyms is returned as a single, space-separated string. 

In [ ]:
from preprocessing.preprocessor import expand_features
df = expand_features(df)
df.iloc[[300000], 7]
# let's check the contents of this entry 
df.iloc[300000, 7]

### TF-IDF Creation and Analysis
We build and use a TF-IDF Matrix as a foundational data structure for our clustering algorithms. 

Term Frequency-Inverse Document Frequency (TF-IDF) is a statistical measure that evaluates how relevant a word is to a document within a larger collection by comparing local frequency to global rarity. Concretely, running the TfidfVectorizer transforms our preprocessed text data into a mathematical grid, where rows are the unique playlists, and our columns are the unique words present in all the names. The values in the matrix cells are scores representing the importance of a specific column's word to a specific row's playlist. This representation allows for a mathematical translation of the strings in a format that supplies crucial context to the clustering models.


#### Building Unique Texts Representation
We will be using the TfidfVectorizer function from the sklearn library in tf-idf matrix creation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

We care only for the entries that are flagged as being contextual from entity filtering, so we create a binary mask for further processing.

In [ ]:
contextual_mask = df['is_contextual'] == True

Using the contextual mask we keep only the non-empty unique values from the expanded features column. 

In [ ]:
unique_texts = df[contextual_mask]['expanded_features'].dropna().unique()

#### Building the TF-IDF Matrix

The imported text extractor and vectorizer is called with parameters:
- **`min_df:`** This tells the vectorizer to completely ignore words that appear in fewer than 5 documents (playlists). This serves to avoid typos or highly personal names that don't help form meaningful clusters.
- **`max_df:`** Words appearing in more than 95% of the documents gets ignored too, this is because words that are extremely common across all playlists act as corpus-specific stop-words, i.e. if every playlist contained 'music' in their name, there would be no discriminative value to help separate the playlists into distinct clusters. While our preprocessing took steps to avoid this as well, the parameters provides a final guardrail.
- **`max_features:`** We restrict the final vocabulary to contain only the top 5,678 most important features. To identify this cutoff, we generated a cumulative importance plot, which ranks all unique features based on their overall contribution to the dataset's total information. By plotting the number of features against the cumulative percentage of variance they capture, an analysis revealed that 80% of the core semantic meaning could be kept in just the top 5,678 features (down from the raw 24,712).

Cumulative importance plot seen below, source-function (`plot_cumulative_importance`):

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from clustering.tf_idf_analysis.tf_idf_analysis import *

In [ ]:
# 1. Initialize the vectorizer with the parameters from the markdown
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
)

# 2. Fit the vocabulary and transform the text into the TF-IDF matrix
tf_idf_matrix = vectorizer.fit_transform(unique_texts)

# 3. The shape of the matrix should be (100793 x 24713)
print(tf_idf_matrix.shape)
plot_cumulative_importance(tf_idf_matrix) # Result will be showcases below


<img src="clustering/tf_idf_analysis/cumulative_importance.png" width="700">

In [ ]:
# 1. Initialize the vectorizer with the restriced max_features parameter
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    max_features=5678,
)
# 2. Fit the vocabulary and transform the text into the TF-IDF matrix
tf_idf_matrix = vectorizer.fit_transform(unique_texts)
# 3. The shape of the matrix should be (100793 x 5678)
print(tf_idf_matrix.shape)

#### Sparsity Analysis

Having just reduced our vocabulary from 24,712 down to 5,678 features, it is highly likely that some outlier playlists only contained words that fell into the discarded "long tail." To understand the impact of this reduction, we run a sparsity analysis.

In [ ]:
# Run only if empty 
import os
sparsity_result_path = "clustering/tf_idf_analysis/" \
"sparsity_analysis_pre_filter.png"
if not os.path.exists(sparsity_result_path):
    plot_sparsity_analysis(tf_idf_matrix)

Sparsity analysis plot seen below, source-function (`plot_sparsity_analysis`):

<img src="clustering/tf_idf_analysis/sparsity_analysis_pre_filter.png" width="800">

The analysis reveals that our matrix is highly sparse (which is expected in TF-IDF text representations), but more importantly, 16097 rows now contain no non-zero entries. Playlists that only contained the least important words have become entirely empty. Because these rows no longer provide any semantic or contextual signal, they are mathematical dead-ends that will only cause errors or skew our geometric clustering algorithms. 

We concluded that these blank playlists must be removed from the dataset before proceeding. The cleaned TF-IDF matrix saved in cache, abstracting the actual removal of these empty rows into the `load_tfidf_matrix` function, which we will use moving forward.

In [ ]:
from clustering.tf_idf_analysis.tf_idf_analysis import load_tfidf_matrix
tfidf_cache_dir = "data/tfidf_cache"
tfidf_matrix, unique_texts, vectorizer = load_tfidf_matrix(tfidf_cache_dir, 
                                                           df, TfidfVectorizer)

## Module 1 - Clustering Methods 
### K-Means (The Paper's Baseline)
K-Means was the method utilized by Pichl et al. We decided to implement this approach to set a baseline we could compare
directly to the original authors’ findings. K-Means requires a predefined number of clusters, k, which we found
analysing the gradient of the Within-Cluster Sum of Squares (WCSS). By applying K-Means, we partition the full-dimensional space using spherical clusters, serving as our standard geometric baseline.

#### WCSS and Finding the Optimal K
To determine the optimal number of clusters ($k$) for our dataset, we employed a quantitative extension of the traditional Elbow Method, based on the methodology by Pichl et al. Rather than relying solely on a visual inspection of the Within-Cluster Sum of Squares (WCSS) slope, we analyzed the discrete derivative, or Delta WCSS, to identify statistically significant reductions in variance. By establishing statistical thresholds for variance reduction, this approach provides a more rigorous, data-driven foundation for our final cluster selection than visual estimation alone.

To implement WCSS we iterate over potential cluster sizes, ranging from 2-99. For each k, we initialize a K-Means model and fit it to our data. We retrieve the kmeans.inertia result which represents the exact desired value: the sum of squared distances of samples to their closest cluster center. We append these and calculate the sequential change in variance to analyze. 

As Pichl et al., we define the threshold for statistical significance as one standard deviation below the mean change, defining the point of diminishing returns. As increasing k provides a natural decrease in variance, we define the threshold to ensure we only consider new clusters worthwhile if they reduce variance by an amount that is statistically larger that the average amount.

In [ ]:
from clustering.kmeans.WCSS.WCSS import load_wcss_results
wcss_dir = "clustering/kmeans/WCSS"
optimal_k, wcss_pngs = load_wcss_results(wcss_dir, 
                                         tfidf_matrix)

In [ ]:
# Using pyplot to make subfigures, plotting them next to each other
wcss = wcss_pngs[0]
wcss_delta = wcss_pngs[1]

fig, axes = plt.subplots(1, 2, figsize=(40, 9))
axes[0].imshow(plt.imread(wcss))
axes[0].axis('off')
axes[1].imshow(plt.imread(wcss_delta))
axes[1].axis('off')
plt.show()

##### Evaluating Variance Reductions (Delta WCSS)
To isolate the exact point of diminishing returns, we examined the Delta WCSS against the aforementioned threshold. While we observe profound initial drops in variance at early stages (e.g., $k=10$ and $k=23$), the graph displays continued, violent oscillation up until the 50-cluster mark.

Following the most unstable segment of the distribution, the data experiences its absolute largest variance reduction at exactly $k=55$, plunging to roughly $-1050$. Crucially, immediately following this final, massive separation, the volatility virtually disappears. From $k=56$ onward, the Delta WCSS rapidly flattens out, hovering consistently above the established threshold without ever crossing it again. By selecting $k=55$ as the optimal number of clusters, we capture the last substantial drop in variance, avoiding the plateau of diminishing returns that follows.







#### Executing KMeans
With the optimal number of clusters ($k=55$) identified, the actual clustering is executed using scikit-learn's KMeans implementation. K-Means aims to divide the dataset into $k$ non-overlapping subsets by minimizing the within-cluster variance. The algorithm operates iteratively (via Lloyd's algorithm): it assigns each playlist vector to the nearest geometric center (centroid), and then updates that centroid to be the mathematical mean of all its assigned points, repeating this process until the centers stop moving.

To implement the model, we instantiate sklearn.cluster.KMeans with the following hyperparameters:
- **`n_clusters=55`**: The target number of distinct groups, derived from our Delta WCSS analysis. This ensures we capture the major structural separations in the corpus without overfitting.
- **`n_init=10`**: A well-known disadvantage of Lloyd’s heuristic algorithm is its sensitivity to initialization as it can easily get trapped in poor local optima. Setting this to 10 forces the algorithm to run 10 complete, independent times with different random starting centroids. The model ultimately keeps the run that achieves the absolute lowest inertia.
- **`max_iter=300`**: Allows each of the 10 runs up to 300 iterations to reach internal convergence. Given the high dimensionality and sparsity of our TF-IDF text space, ensuring sufficient iterations guarantees that the centroids have fully stabilized before termination.
- **`random_state=RANDOM_SEED`**: Fixes the underlying random number generator to ensure perfectly reproducible results across environments.

After fitting the model, a dictionary maps the resulting cluster labels back to the original DataFrame based on the expanded text features. 

In [ ]:
from notebook_helper import execution_pipeline

In [ ]:
from clustering.kmeans.kmeans_clustering import KMeansClustering

Intantiate KMeansClustering with the optimal number of clusters and other parameters.

In [ ]:
# Intantiate KMeansClustering with the optimal number of clusters and other parameters
KMeans = KMeansClustering(k=optimal_k, max_iter=300, 
                          n_init=10, random_state=RANDOM_SEED)
execution_pipeline(KMeans, df, unique_texts, tfidf_matrix)

### Birch (Hierachical Clustering)
To address the potential limitations of K-Means’ rigid, global partitioning, we implemented BIRCH (Balanced Iterative
Reducing and Clustering using Hierarchies).

BIRCH operates in two distinct phases that structurally mirror the nested nature of music (e.g., sub-genres). 
Playlists are processed incrementally, constructing a specialized data structure called a Clustering Feature (CF) Tree. Data points are summarized into CFs which represent a tightly packed sphere of points. If a playlist vector falls within the radius of an existing sphere, it is absorbed into that sphere, if it doesn't, a new CF is created. This creates clusters, but not necessarily $k$. Therefore an agglomerative hierarchical clustering algorithm is run on the centroids marge the closest pairs until there are $k$ clusters.

K-Means is known to be sensitive to noise, often letting outliers/noise pull centroids off center, as they’re forced to assign every point to a cluster. The bottom-up approach of BIRCH ensures that the final macro-clusters are built from pure semantic building blocks rather than forced global averages. Furthermore, the final agglomerative merge allows for non-spherical clusters, unlike k-means. By finding clusters from highly specific sub-contexts, BIRCH allows us to test if a hierarchical structural definition better captures the semantic reality of human musical curation.

#### Executing Birch
To implement BIRCH we instantiate the sklearn.cluster.Birch model with three hyperparameters.
- **`k=55`**: The amount of desired clusters. Derived from WCSS, chosen such that we can compare results with K-means.
- **`Thereshold=0.9`**: Controls the maximum radius of a sub-cluster. As our TF-IDF matrix creates a high-dimensional sparse space, the points will naturally be far apart. A threshold of 0.9 aims to be forgiving enough to group slight varying text structures (e.g., "workout mix" and "gym playlist"), while still separating distinct broader contexts.
- **`Branching Factor=25`**: Defines the width limit of the tree. 25 was chosen as a moderates alternative to the default 50, which aims to mirror the deeply nested music taxonomy. 
We thereafter call the fit and predict methods, corresponding to the aforementioned process.
- **`random_state=RANDOM_SEED`**: Fixes the underlying random number generator to ensure perfectly reproducible results across environments.

In [ ]:
from clustering.birch.birch_clustering import BirchClustering

Initializing BirchClustering instance.

In [ ]:
# Initialize BirchClustering instance
Birch = BirchClustering(k=55, threshold=0.9, 
                        branching_factor=25, batch_size=1000)
execution_pipeline(Birch, df, unique_texts, tfidf_matrix)

### Evaluation
To validate whether our discovered clusters represent meaningful musical contexts, we implemented a recommendation
evaluation pipeline directly mirroring the methodology of Pichl et al. Since we don’t have a ground truth, we
evaluate the clusters based on their utility in a Collaborative Filtering scenario.

To do this, we restrict the collaborative filtering within each cluster. The aim is hence to analyze whether the
boundaries make semantic sense. We simulate a recommendation scenario by splitting songs within a playlist into
test and training dictionaries. The algorithm is then tasked with predicting these missing tracks by utilizing similar
playlists within the target’s cluster. We quantify the relevance of each potential neighbour using Jaccard Similarity. Let
T represent the set of visible tracks in the target playlist, and N represent the set of tracks in a neighbouring playlist. The similarity score is calculated as the intersection over the union:

$$J(T, N) = \frac{|T \cap N|}{|T \cup N|}$$

For each neighbour the algorithm isolates new tracks, where they are assigned a weight equal to the sum of Jaccard
similarities of the neighbors that provided it. These tracks are thereafter sorted in descending order and we evaluate
how many of the originally "masked" tracks appear at the very top of this ranked list using the $F_{0.1}$ metric. Ultimately,
a higher score proves that the clustering algorithm successfully grouped pure, cohesive musical contexts, allowing the
collaborative filter to draw from a highly relevant neighbourhood.

The performance of the recommendation models is illustrated in the graph, which compares the $F_{0.1}$-measure (y-axis) against the recommendation density $p$ (x-axis), representing the fraction of the test set size used to generate recommendations. The "top-k" lines (Top-1, Top-5, Top-10) represent the average performance of clusters that were ranked in descending order based on their computed $F_{0.1}$ scores.


In [ ]:
from notebook_helper import evaluation_pipeline

#### Evaluating K-Means

In [ ]:
k_means_report_out = evaluation_pipeline(KMeans, df, 
                                         unique_texts, tfidf_matrix)

#### Evaluating BIRCH

In [ ]:
birch_report_out = evaluation_pipeline(Birch, df, 
                                       unique_texts, tfidf_matrix)

#### Plotting the $F_{0.1}$ Results

In [ ]:
import os

# Plotting the results side by side for comparison
k_means_55_path = os.path.join(k_means_report_out, 
                               "f01_comparison.png")
birch_path = os.path.join(birch_report_out, 
                          "f01_comparison.png")

# Plotting them side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread(k_means_55_path))
axes[0].set_title(f"{KMeans.algo_name} F1-Score Comparison")
axes[0].axis('off')
axes[1].imshow(plt.imread(birch_path))
axes[1].set_title(f"{Birch.algo_name} F1-Score Comparison")
axes[1].axis('off')
plt.tight_layout()
plt.show()

#### Overall Predictive Trends
The results across both implementations exhibit an intuitive hierarchy: evaluating only the absolute highest-scoring cluster (Top-1) yields the highest predictive accuracy. As we average performance across wider brackets (Top-5, Top-10, and Top-all), the overall score predictably degrades as lower-signal clusters are introduced. Generally, increasing $p$ causes the score to decline as the system is forced to fill a larger quota with tracks of diminishing relevance. Both K-Means and BIRCH occasionally however show performance gains when moving from $p=0.1$ to $0.3$ and occasionally beyond. This suggests that a very small $p$ may be too restrictive; by $p=0.3$, the algorithm is allowed enough "guesses" to capture relevant tracks that were ranked highly but not first.


#### Comparison to the Original Findings
When comparing our findings to the baseline established by Pichl et al., two major distinctions emerge. First, both of our models demonstrate a higher baseline of accuracy. Second, the original paper's results exhibit a significant drop in performance immediately after $p=0.6$, whereas out models maintain a more smooth gradual decay. We attribute the discrepancy to our enhanced preprocessing pipeline.
By applying more strict entity and stop word filtering prior to our TF-IDF vectorizations, we significantly increased the semantic signal-to-noise ratio.

#### K-Means vs. Birch
The empirical disparity between K-means (max $F_{0.1} \approx 0.65$) and BIRCH (max $F_{0.1} \approx 0.75$) validates our core hypothesis. K-means optimizes a global objective function that enforces a rigid, spherical cluster geometry. When applied to our high-dimensional sparse TF-IDF matrix, this representative-based approach is highly susceptible to noise, as outliers disproportionately distort the global centroids, diluting the intra cluster similarities. BIRCH on the other hand uses a structural paradigm that naturally quarantines semantic noise and topological outliers into isolated leaf nodes, preventing them from skewing the broader centroids. This results in the evaluation algorithm drawing from a search space of higher semantic purity. The results confirm that a hierarchical model is better suited to model the boundaries of human musical taxonomy.



### Hypothesis For Limitations and Potential Improvements


While our findings demonstrate improved performance over the baseline, both K-Means and BIRCH suffer from a persistent "gravity well" effect, where a single cluster absorbs a large percentage of the dataset:

In [ ]:
# Plotting the gravity wells
k_means_55_path = os.path.join(k_means_report_out, 
                               "cluster_distribution.png")
birch_path = os.path.join(birch_report_out, 
                          "cluster_distribution.png")

# Plotting them side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread(k_means_55_path))
axes[0].set_title(f"{KMeans.algo_name} Gravity Well Comparison")
axes[0].axis('off')
axes[1].imshow(plt.imread(birch_path))
axes[1].set_title(f"{Birch.algo_name} Gravity Well Comparison")
axes[1].axis('off')
plt.tight_layout()
plt.show()

Based on our analysis, we hypothesize the following limitations and propose corresponding improvements:

#### The Curse of Dimensionality
**Limitation:** In our 5,000-dimensional sparse TF-IDF text space, standard geometric distance metrics (like Euclidean distance) lose their discriminative power. High volumes of orthogonal vectors cause points to become nearly equidistant, making rigid, spherical cluster boundaries arbitrary. This reduces the effectiveness of the two clustering algorithms.

**Improvement (Subspace Clustering):** We hypothesize that true contextual boundaries only exist in specific, lower-dimensional slices of the data where generic terms are ignored. Implementing a Subspace Clustering algorithm would allow the model to find local clusters across hidden subsets of features.

#### Drowning in Generic Vocabulary
**Limitation:** TF-IDF evaluates word importance on a global dataset scale. Although significant preprocessing was implemented, generic textual overlap often drowns out the niche, localized signals required to cleanly separate overlapping sub-genres.

**Improvement (Dimensionality Reduction):** Applying algorithms like Truncated SVD (LSA) or UMAP prior to geometric clustering would condense co-occurring terms into latent topics. This reduces the empty space in our matrix, allowing the algorithms to group playlists by their actual thematic context instead of exact word matches.

#### Moving Beyond Raw Vocabulary Intersections
**Limitation:** The TF-IDF methodology strictly requires exact text intersections. Contextually identical playlists such as "workout mix" and "gym motivation" remain orthogonal if they share no exact words. Currently, our pipeline is highly over-reliant on the Feature Expansion step, hoping that WordNet successfully injects a shared synonym to bridge this gap. If this brute-force expansion fails to produce an exact string match, the algorithms are forced to group the playlists based on distant, weak overlaps (contributing to the gravity well).
 
**Improvement (Dense Vector Embeddings):** Replacing the TF-IDF representation with dense semantic language models (e.g., Word2Vec, GloVe, or BERT embeddings) would allow the clustering mechanisms to draw boundaries based on actual human inference rather than raw vocabulary intersections.

### Investigating Dimensionality Reduction + KMeans
Of these three proposed improvements, the investigation of dimensionality reduction, specifically Truncated Singular Value Decomposition (SVD), seems promising as it directly addresses two of our core limitations: the curse of high dimensional sparsity and the effect of generic vocabulary.

Instead of operating in the high-dimensional sparse space where K-means notoriously struggles, SVD projects the data into a low-dimensional dense latent semantic space. This transformation should eliminate noise, sharpen cluster boundaries, and improve computational efficiency. By compressing our 5,678 sparse features into dense latent features, we can isolate whether K-Means' poor performance stems from its algorithmic assumptions or from the high-dimensional representation itself.


#### Establishing the Optimal Latent Dimensions
To find the optimal feature reduction for SVD we can look at the Cumulative Explained Variance Ratio. This process mirrors the cumulative importance analysis performed during the initial TF-IDF creation. During this setup, we performed *Feature Selection*, ranking literal words by their frequency and dropping the rarest ones, reducing our vocabulary to the top 5,678 terms that held 80% of the information. By contrast, SVD performs *Feature Extraction*, mathematically blending all 5,678 sparse features into new dimensions (representing underlying thematic topics). The Explained Variance measures how much of the dataset's original structure is preserved within these new components.

By plotting the cumulative variance, much like our earlier TF-IDF plot, we can identify the precise number of dimensions required to capture a significant threshold of the total semantic information (e.g., 80%).


In [33]:
from scripts.svd_info_loss import plot_svd_variance
plot_svd_variance(tfidf_matrix, random_seed=RANDOM_SEED)

Our cumulative variance analysis indicates that 656 components are required to retain 80% of the dataset's mathematical variance. However, after empirical testing, we discovered that the resulting recommendations proved worse than the original K-Means baseline. We attribute this to the fact that cumulative variance thresholds are traditionally designed for continuous numerical data; in text data, the variance captured beyond the first few hundred components largely consists of semantic noise rather than useful taxonomy. In Latent Semantic Analysis (LSA), standard practice strictly caps components between 100 and 300 (Landauer & Dumais, 1997). Our findings confirm this, and the execution below demonstrates the improved results using an optimal 200 components.

#### Executing K-Means in the Latent Space
With the optimal parameter established, we instantiate our pipeline to first project the TF-IDF matrix into the 200-dimensional dense semantic space. We then immediately pass this noise-reduced output into a new K-Means model, maintaining $k=55$ and our previous initialization hyperparameters ($n\_init=10$, $max\_iter=300$), to perform the final geometric clustering.

In [ ]:
from clustering.svdkmeans.svdkmeans import SVDKMeansClustering
SVDKMeans = SVDKMeansClustering(k=optimal_k, n_components=200, 
                                max_iter=300, n_init=10, 
                                random_state=RANDOM_SEED)

In [ ]:
execution_pipeline(SVDKMeans, df, unique_texts, tfidf_matrix)

#### Comparative Analysis: Baseline vs. SVD-Enhanced K-Means

In [ ]:
SVDKmeans_report_path = evaluation_pipeline(SVDKMeans, df, 
                                            unique_texts, tfidf_matrix)

In [ ]:
# Plotting kmeans + cdist vs SVDKMeans + cdist
k_means_55_path = os.path.join(k_means_report_out, "f01_comparison.png")
svd_k_means_path = os.path.join(SVDKmeans_report_path, "f01_comparison.png")
# Plotting them side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread(k_means_55_path))
axes[0].set_title(f"{KMeans.algo_name} F1-Score Comparison")
axes[0].axis('off')
axes[1].imshow(plt.imread(svd_k_means_path))
axes[1].set_title(f"{SVDKMeans.algo_name} F1-Score Comparison")
axes[1].axis('off')
plt.tight_layout()
plt.show()

The execution of K-Means on the 200-dimensional latent space resulted in noticeable enhancement of cluster cohesion, as proved by the increased recommendation $F_{0.1}$ scores. This empirically validates our dual hypotheses regarding the "Curse of Dimensionality" and the limitation of heavily generic vocabulary. By projecting the dataset into a lower-dimensional semantic space, SVD grouped co-occurring terms into latent topics. This effectively discarded the long tail of textual noise and allowed the algorithm to group contexts without requiring exact vocabulary intersections.

Notably, this dimensionality reduction allowed K-Means to close the performance gap with the more flexible Hierarchical model (BIRCH). This indicates that standard geometric clustering is capable of defining boundaries in human-generated musical contexts, provided the underlying ambient space is dense and analytically pure. Since BIRCH's initial Clustering Feature (CF) Tree construction also relies on Euclidean distance, which is uninformative in high dimensions, we predict a similar structural improvement when applying SVD to the inputs given to BIRCH.
